In [3]:
# Uninstall CPU-only TensorFlow and install GPU version
import sys
!{sys.executable} -m pip uninstall -y tensorflow tensorflow-intel
!{sys.executable} -m pip install tensorflow==2.15.0

Found existing installation: tensorflow 2.13.1
Uninstalling tensorflow-2.13.1:
  Successfully uninstalled tensorflow-2.13.1
Found existing installation: tensorflow-intel 2.13.1
Uninstalling tensorflow-intel-2.13.1:
  Successfully uninstalled tensorflow-intel-2.13.1


You can safely remove it manually.


  Using cached tensorflow-2.15.0-cp311-cp311-win_amd64.whl.metadata (3.6 kB)
  Using cached tensorflow_intel-2.15.0-cp311-cp311-win_amd64.whl.metadata (5.1 kB)
  Using cached ml_dtypes-0.2.0-cp311-cp311-win_amd64.whl.metadata (20 kB)
  Using cached wrapt-1.14.2-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached tensorboard-2.15.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached tensorflow_estimator-2.15.0-py2.py3-none-any.whl.metadata (1.3 kB)
  Using cached keras-2.15.0-py3-none-any.whl.metadata (2.4 kB)
   ---------------------------------------- 0.0/300.9 MB ? eta -:--:--
   ---------------------------------------- 0.8/300.9 MB 6.6 MB/s eta 0:00:46
   ---------------------------------------- 2.9/300.9 MB 9.9 MB/s eta 0:00:31
   ---------------------------------------- 3.7/300.9 MB 7.8 MB/s eta 0:00:39
    --------------------------------------- 5.0/300.9 MB 6.6 MB/s eta 0:00:46
    --------------------------------------- 6.0/300.9 MB 6.4 MB/s eta 0:00:47
    ---------------

  You can safely remove it manually.


In [ ]:
# Imports
import os, warnings
import matplotlib.pyplot as plt
from matplotlib import gridspec

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Reproducability
def set_seed(seed=31415):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
set_seed(31415)

# Set Matplotlib defaults
plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large',
       titleweight='bold', titlesize=18, titlepad=10)
plt.rc('image', cmap='magma')
warnings.filterwarnings("ignore") # to clean up output cells
# Load training and validation sets (using 224x224 for VGG16's native input size)

# Load training and validation sets
ds_train_ = image_dataset_from_directory(
    'C:/Users/User/Downloads/train',
    image_size=[224, 224],
    label_mode='binary',
    batch_size=32,  # Reduced batch size for larger images
    interpolation='nearest',
    batch_size=64,
    shuffle=True,
)
ds_valid_ = image_dataset_from_directory(
    'C:/Users/User/Downloads/valid',
    image_size=[224, 224],
    label_mode='binary',
    batch_size=32,
    interpolation='nearest',
    batch_size=64,
    shuffle=False,
# Data Augmentation Layer
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

# Data Pipeline
def convert_to_float(image, label):
    image = tf.image.convert_image_dtype(image, dtype=tf.float32)
    return image, label

def augment(image, label):
    image = data_augmentation(image, training=True)
    return image, label

AUTOTUNE = tf.data.experimental.AUTOTUNE

ds_train = (
    .prefetch(buffer_size=AUTOTUNE))

    ds_train_
)    .prefetch(buffer_size=AUTOTUNE)

    .map(convert_to_float)    .cache()

    .map(augment)  # Apply augmentation    .map(convert_to_float)

    .cache()    ds_valid_

    .prefetch(buffer_size=AUTOTUNE)ds_valid = (
)

Found 5117 files belonging to 2 classes.
Found 5051 files belonging to 2 classes.


In [6]:
import matplotlib.pyplot as plt

In [ ]:
from tensorflow.keras.applications import VGG16

pretrained_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
pretrained_base.trainable = False

58889256/58889256 [==============================] - 19s 0us/step


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    pretrained_base,
    layers.GlobalAveragePooling2D(),  # More efficient than Flatten
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(128, activation='relu'),])

    layers.Dropout(0.3),    layers.Dense(1, activation='sigmoid'),

In [ ]:
# Compile with advanced optimizer and metrics
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=[
        'binary_accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc'),
    ],
)


# Callbacks for better training)

callbacks = [    verbose=1,

    tf.keras.callbacks.EarlyStopping(    callbacks=callbacks,

        monitor='val_loss',    epochs=50,

        patience=5,    validation_data=ds_valid,

        restore_best_weights=True,    ds_train,

        verbose=1history = model.fit(

    ),

    tf.keras.callbacks.ReduceLROnPlateau(]

        monitor='val_loss',    )

        factor=0.5,        verbose=1

        patience=3,        save_best_only=True,

        min_lr=1e-7,        monitor='val_loss',

        verbose=1        'best_model.keras',

    ),    tf.keras.callbacks.ModelCheckpoint(

In [ ]:
import pandas as pd

# Plot training history
history_frame = pd.DataFrame(history.history)


fig, axes = plt.subplots(2, 2, figsize=(15, 10))plt.show()

plt.tight_layout()

# Loss

axes[0, 0].plot(history_frame['loss'], label='Training Loss')axes[1, 1].grid(True)

axes[0, 0].plot(history_frame['val_loss'], label='Validation Loss')axes[1, 1].legend()

axes[0, 0].set_title('Model Loss')axes[1, 1].set_ylabel('Recall')

axes[0, 0].set_xlabel('Epoch')axes[1, 1].set_xlabel('Epoch')

axes[0, 0].set_ylabel('Loss')axes[1, 1].set_title('Model Recall')

axes[0, 0].legend()axes[1, 1].plot(history_frame['val_recall'], label='Validation Recall')

axes[0, 0].grid(True)axes[1, 1].plot(history_frame['recall'], label='Training Recall')

# Recall

# Accuracy

axes[0, 1].plot(history_frame['binary_accuracy'], label='Training Accuracy')axes[1, 0].grid(True)

axes[0, 1].plot(history_frame['val_binary_accuracy'], label='Validation Accuracy')axes[1, 0].legend()

axes[0, 1].set_title('Model Accuracy')axes[1, 0].set_ylabel('Precision')

axes[0, 1].set_xlabel('Epoch')axes[1, 0].set_xlabel('Epoch')

axes[0, 1].set_ylabel('Accuracy')axes[1, 0].set_title('Model Precision')

axes[0, 1].legend()axes[1, 0].plot(history_frame['val_precision'], label='Validation Precision')

axes[0, 1].grid(True)axes[1, 0].plot(history_frame['precision'], label='Training Precision')

# Precision

In [ ]:
# Evaluate on validation set with detailed metrics
y_true = []
y_pred = []

for images, labels in ds_valid:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend((predictions > 0.5).astype(int).flatten())

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Class 0', 'Class 1']))

In [ ]:
# Optional: Fine-tune the model by unfreezing top layers
print("Fine-tuning the model...")

# Unfreeze the last 4 layers of VGG16
pretrained_base.trainable = True
for layer in pretrained_base.layers[:-4]:
    layer.trainable = False

# Recompile with lower learning rate for fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=[
        'binary_accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc'),
    ],
)

# Fine-tune for additional epochs
history_finetune = model.fit(
    ds_train,
    validation_data=ds_valid,
    epochs=20,
    callbacks=callbacks,
    verbose=1,
)

print("Fine-tuning complete!")

In [1]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.config.list_physical_devices('GPU'))

# Additional debugging
import os
print("\nCUDA_VISIBLE_DEVICES:", os.environ.get('CUDA_VISIBLE_DEVICES', 'Not set'))
print("TensorFlow version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())


Num GPUs Available:  0
[]

CUDA_VISIBLE_DEVICES: Not set
TensorFlow version: 2.15.0
Built with CUDA: False
